# Pipeline for Variant Analysis using Gene Coding Sequence
Using this pipeline I tried understand how different types of variants of the human beta-globin gene (HBB) coding sequence affect the translated protein and its calculated physicochemical properties.

### 1. Install Biopython
Biopython is used for retrieving, reading, manipulating, translating, and analysing biological sequences.

In [1]:
# Install the Biopython library to interact with NCBI databases
!pip install biopython


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. Retrieve the HBB reference transcript from NCBI
This cell uses **Biopython Entrez** to fetch the human beta-globin *(HBB)* RefSeq mRNA record `NM_000518.5` from the NCBI Nucleotide database in GenBank format. It stores the returned record as text and displays an excerpt so that its annotations and sequence-record structure can be inspected.

In [1]:
# Import the Entrez module from Bio for querying NCBI
from Bio import Entrez

# Provide an email address to NCBI as required by their policy
Entrez.email = "<your_email>"

# Define the RefSeq accession number to retrieve
accession_id = "NM_000518.5"

# Fetch the record from the 'nucleotide' database in GenBank ('gb') format as text
with Entrez.efetch(db="nucleotide", id=accession_id, rettype="gb", retmode="text") as handle:
    # Read the retrieved record content
    refseq_data = handle.read()

# Print the first 1000 characters of the GenBank record to view its structure and details
print(refseq_data[:1000])

# Print the first 1000 characters of the GenBank record to view its structure and details
print(refseq_data[:1000])

LOCUS       NM_000518                628 bp    mRNA    linear   PRI 06-SEP-2026
DEFINITION  Homo sapiens hemoglobin subunit beta (HBB), mRNA.
ACCESSION   NM_000518
VERSION     NM_000518.5
KEYWORDS    RefSeq; MANE Select.
SOURCE      Homo sapiens (human)
  ORGANISM  Homo sapiens
            Eukaryota; Metazoa; Chordata; Craniata; Vertebrata; Euteleostomi;
            Mammalia; Eutheria; Euarchontoglires; Primates; Haplorrhini;
            Catarrhini; Hominidae; Homo.
REFERENCE   1  (bases 1 to 628)
  AUTHORS   Civettini,I., Zappaterra,A., Corti,P., Messina,A., Aroldi,A.,
            Biondi,A., Cavalca,F., Crippa,V., Crosti,F., Ferrari,G.M.,
            Malighetti,F., Mologni,L., Piperno,A., Ramazzotti,D., Scollo,C.,
            Zambon,A., Rossi,F. and Gambacorti-Passerini,C.
  TITLE     Hb Monza: A novel extensive HBB duplication with preserved
            alpha-beta subunit interaction and unstable hemoglobin phenotype
  JOURNAL   Med 6 (4), 100554 (2025)
   PUBMED   39701095
REFERENCE

### 3. Parse the GenBank record and extract the mRNA sequence
This cell uses **SeqIO** to parse the downloaded GenBank record. It extracts the complete mRNA sequence and displays both its length and nucleotide sequence. The resulting `mrna_sequence` object is used in the next step to obtain the coding region.

In [2]:
# Import io to handle the string data as a file stream and SeqIO to parse the GenBank format
import io
from Bio import SeqIO

# Read the GenBank format record using SeqIO.read
record = SeqIO.read(io.StringIO(refseq_data), "genbank")

# Extract the full mRNA sequence
mrna_sequence = record.seq

# Print the total length of the retrieved mRNA
print(f"Total mRNA Sequence Length: {len(mrna_sequence)} bp\n")

# Print the entire mRNA sequence
print("Full mRNA Sequence:")
print(mrna_sequence)

Total mRNA Sequence Length: 628 bp

Full mRNA Sequence:
ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACCATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGACAACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGATCCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAATGCCCTGGCCCACAAGTATCACTAAGCTCGCTTTCTTGCTGTCCAATTTCTATTAAAGGTTCCTTTGTTCCCTAAGTCCAACTACTAAACTGGGGGATATTATGAAGGGCCTTGAGCATCTGGATTCTGCCTAATAAAAAACATTTATTTTCATTGCAA


### 4. Extract the HBB coding sequence (CDS)
This cell selects the HBB coding region from the mRNA using the specified biological coordinates, positions 51–494. It converts the 1-based inclusive coordinates into Python's 0-based slicing convention, then displays the extracted CDS and its length.

In [3]:
# Define the biological 1-based start and end coordinates of the CDS
start_pos = 51
end_pos = 494

# Convert 1-based biological coordinates to 0-based Python slice indices
# Position 51 corresponds to index 50, and position 494 is inclusive (so index 494 for slicing)
cds_slice = mrna_sequence[start_pos - 1 : end_pos]

# Print the sliced CDS region
print(f"CDS Sequence (Positions {start_pos} to {end_pos}, length: {len(cds_slice)} bp):")
print(cds_slice)

CDS Sequence (Positions 51 to 494, length: 444 bp):
ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGACAACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGATCCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAATGCCCTGGCCCACAAGTATCACTAA


### 5. Load and validate the reference CDS FASTA file
For this analysis, I already had a FASTA file and for the analyses beyond this point, I use the reference CDS in this file. This cell checks whether the reference CDS FASTA file is already available and, if not, provides an upload method suited to the notebook environment. In **Google Colab**, it uses `google.colab.files.upload()`. In local **Jupyter**, it displays an `ipywidgets` file-upload control. For the local option, select the file and run the cell again so the uploaded contents can be saved under the expected filename. The cell then reads the FASTA record and performs basic validation: sequence count, length divisibility by three, start codon, terminal stop codon, internal in-frame stop codons, and GC fraction.

In [4]:
# Import the GC function from Bio.SeqUtils to calculate GC percentage
from Bio.SeqUtils import gc_fraction
# Import modules for file handling and environment detection
import os
import sys

# Define the filename expected by the rest of the pipeline
fasta_filename = "HBB_reference_CDS.fasta"

# Detect whether this notebook is running in Google Colab
try:
    import google.colab
    running_in_colab = True
except ImportError:
    running_in_colab = False

# Check whether the FASTA file is already available
upload_ready = os.path.exists(fasta_filename)

if upload_ready:
    print(f'"{fasta_filename}" already loaded in the current working directory.')

elif running_in_colab:
    # Import files from google.colab to handle interactive file uploads
    from google.colab import files

    print(f"Please select and upload '{fasta_filename}':")
    uploaded_files = files.upload()

    if fasta_filename in uploaded_files:
        upload_ready = True
    elif len(uploaded_files) == 1:
        # Accept a single uploaded FASTA file even if its original name differs
        uploaded_name = next(iter(uploaded_files))
        with open(fasta_filename, "wb") as output_file:
            output_file.write(uploaded_files[uploaded_name])
        upload_ready = True
    else:
        print(f"Upload exactly one file, preferably named '{fasta_filename}'.")

else:
    # Local Jupyter: use an ipywidgets upload control.
    # Keep the widget in the kernel's global namespace so its uploaded contents remain available when this cell is run a second time.
    try:
        import ipywidgets as widgets
        from IPython.display import display

        if "_hbb_fasta_upload_widget" not in globals():
            _hbb_fasta_upload_widget = widgets.FileUpload(
                accept=".fasta,.fa,.fna",
                multiple=False,
                description="Choose FASTA"
            )

        upload_widget = _hbb_fasta_upload_widget
        uploaded_value = upload_widget.value

        if not uploaded_value:
            print(
                "Local Jupyter detected. Choose the FASTA file using the "
                "upload control below, then run this cell again."
            )
            display(upload_widget)
        else:
            # ipywidgets 8 returns a tuple of file dictionaries.
            # ipywidgets 7 returns a dictionary keyed by filename.
            if isinstance(uploaded_value, dict):
                uploaded_file = next(iter(uploaded_value.values()))
            else:
                uploaded_file = uploaded_value[0]

            uploaded_content = uploaded_file["content"]
            uploaded_content = bytes(uploaded_content)

            with open(fasta_filename, "wb") as output_file:
                output_file.write(uploaded_content)

            print(f"Uploaded file saved as '{fasta_filename}'.")
            upload_ready = True
            del globals()["_hbb_fasta_upload_widget"]

    except ImportError:
        print(
            "Local Jupyter upload requires ipywidgets. Install it with "
            "`pip install ipywidgets`, restart or refresh the notebook, "
            "and run this cell again."
        )

# Continue with sequence validation only after the file is available
if upload_ready:
    # Read and load all records from the FASTA file into a list
    records = list(SeqIO.parse(fasta_filename, "fasta"))

# Confirm that the FASTA file contains exactly one DNA sequence
print(f"Number of sequences in file: {len(records)}")
if len(records) == 1:
    # Extract the single record
    record = records[0]
    # Extract the sequence object
    seq = record.seq
    # Calculate the length of the sequence
    seq_len = len(seq)

    # 1. Print sequence length and check if it is divisible by three
    print(f"Sequence Length: {seq_len} bp (Divisible by 3: {seq_len % 3 == 0})")

    # 2. Check if the sequence starts with the start codon 'ATG'
    starts_with_atg = seq.startswith("ATG")
    print(f"Starts with ATG: {starts_with_atg}")

    # 3. Check if it ends with an in-frame stop codon ('TAA', 'TAG', 'TGA')
    stop_codons = {"TAA", "TAG", "TGA"}
    ending_codon = str(seq[-3:])
    ends_with_stop = ending_codon in stop_codons
    print(f"Ends with in-frame stop codon ({ending_codon}): {ends_with_stop}")

    # 4. Check for any internal in-frame stop codons
    internal_stops = []
    # Iterate through codons from start up to the second-to-last codon
    for i in range(0, seq_len - 3, 3):
        codon = str(seq[i:i+3])
        if codon in stop_codons:
            # Record the codon and its 1-based position if a stop is found
            internal_stops.append((codon, i + 1))

    # Print status of internal stop codons
    if len(internal_stops) == 0:
        print("Internal in-frame stop codons: None")
    else:
        print(f"Internal in-frame stop codons found: {internal_stops}")

    # 5. Calculate and print the GC percentage of the sequence
    gc_percentage = gc_fraction(seq)
    print(f"GC Content Percentage: {gc_percentage:.2f}%")
else:
    print("Error: The file does not contain exactly one sequence.")

"HBB_reference_CDS.fasta" already loaded in the current working directory.
Number of sequences in file: 1
Sequence Length: 444 bp (Divisible by 3: True)
Starts with ATG: True
Ends with in-frame stop codon (TAA): True
Internal in-frame stop codons: None
GC Content Percentage: 0.56%


### 6. Search NCBI for human HBB RefSeq mRNA records
I used a specific reference accession to fetch a HBB RefSeq mRNA record earlier, but wish to determine if more HBB transcript records are available in NCBI.

This cell runs an Entrez search restricted to human (*Homo sapiens*) HBB RefSeq mRNA records. It retrieves the matching NCBI record IDs and, when results are available, uses Entrez summary records to display their accession versions and titles.

In [5]:
# Define search term targeting human HBB RefSeq mRNA variants
search_term = "HBB[Gene] AND Homo sapiens[Organism] AND srcdb_refseq[prop] AND biomol_mrna[prop]"

# Execute search query to find database IDs
with Entrez.esearch(db="nucleotide", term=search_term, retmode="xml") as handle:
    search_results = Entrez.read(handle)

# Retrieve the list of IDs from search results
ncbi_ids = search_results["IdList"]
print(f"Found {len(ncbi_ids)} NCBI RefSeq transcript IDs for HBB.\n")

# Fetch details for the found IDs if list is not empty
if ncbi_ids:
    with Entrez.esummary(db="nucleotide", id=",".join(ncbi_ids)) as summary_handle:
        summary_records = Entrez.read(summary_handle)
        # Iterate through summaries to print accession IDs and names
        for record in summary_records:
          print(f"Variant ID: {record['AccessionVersion']} | {record['Title']}")

Found 1 NCBI RefSeq transcript IDs for HBB.

Variant ID: NM_000518.5 | Homo sapiens hemoglobin subunit beta (HBB), mRNA


### 7. Create test variants and save them as a CSV table
This cell creates a DataFrame containing 12 HBB sequence variants: eight single-nucleotide substitutions, one single-base insertion, one single-base deletion, one three-base insertion, and one three-base deletion. Each row records a variant ID, variant type, position, reference allele, and alternate allele. The table is saved as `variants.csv` and displayed for inspection.

In [6]:
import pandas as pd

# Define the updated list of 12 variants matching the reference sequence (seq) accurately
# We ensure all position-to-nucleotide relationships are perfectly correct to satisfy validation
variants_data = [
    # Substitutions (Replaces the nucleotide at position)
    {"Variant_ID": "V01", "Type": "Substitution", "Position": 21, "Ref_Allele": "G", "Alt_Allele": "A"},
    {"Variant_ID": "V02", "Type": "Substitution", "Position": 20, "Ref_Allele": "A", "Alt_Allele": "T"},
    {"Variant_ID": "V03", "Type": "Substitution", "Position": 19, "Ref_Allele": "G", "Alt_Allele": "T"},
    {"Variant_ID": "V04", "Type": "Substitution", "Position": 75, "Ref_Allele": "T", "Alt_Allele": "C"},
    {"Variant_ID": "V05", "Type": "Substitution", "Position": 9, "Ref_Allele": "T", "Alt_Allele": "A"},
    {"Variant_ID": "V06", "Type": "Substitution", "Position": 60, "Ref_Allele": "C", "Alt_Allele": "A"},
    {"Variant_ID": "V07", "Type": "Substitution", "Position": 120, "Ref_Allele": "G", "Alt_Allele": "A"},
    {"Variant_ID": "V08", "Type": "Substitution", "Position": 180, "Ref_Allele": "G", "Alt_Allele": "A"},
    # Insertion (Adds alt immediately after the nucleotide at position)
    {"Variant_ID": "V09", "Type": "Insertion", "Position": 100, "Ref_Allele": "", "Alt_Allele": "A"},
    # Deletion (Removes the ref sequence starting at position)
    {"Variant_ID": "V10", "Type": "Deletion", "Position": 160, "Ref_Allele": "G", "Alt_Allele": ""},
    # Insertion (Adds alt immediately after the nucleotide at position)
    {"Variant_ID": "V11", "Type": "Insertion", "Position": 205, "Ref_Allele": "", "Alt_Allele": "GCT"},
    # Deletion (Removes the ref sequence starting at position)
    {"Variant_ID": "V12", "Type": "Deletion", "Position": 250, "Ref_Allele": "GGC", "Alt_Allele": ""}
]

# Convert the list of dictionaries into a pandas DataFrame
df_variants = pd.DataFrame(variants_data)

# Save the DataFrame to a CSV file named 'variants.csv'
df_variants.to_csv("variants.csv", index=False)

# Display the newly created variants table
display(df_variants)

,Variant_ID,Type,Position,Ref_Allele,Alt_Allele
0,V01,Substitution,21,G,A
1,V02,Substitution,20,A,T
2,V03,Substitution,19,G,T
3,V04,Substitution,75,T,C
4,V05,Substitution,9,T,A
5,V06,Substitution,60,C,A
6,V07,Substitution,120,G,A
7,V08,Substitution,180,G,A
8,V09,Insertion,100,,A
9,V10,Deletion,160,G,


### 8. Organise the input and output directory structure
This cell creates the main project directory and its *Input files* and *Output files* subdirectories. If the reference CDS FASTA and variant CSV are present in the current session, it copies them into the corresponding project folders.

In [7]:
import shutil

# Create the main directory 'Variant-Analysis-for-HBB-Gene'
os.makedirs('Variant-Analysis-for-HBB-Gene', exist_ok=True)

# Create 'Input files' and 'Output files' subdirectories
os.makedirs('Variant-Analysis-for-HBB-Gene/Input files', exist_ok=True)
os.makedirs('Variant-Analysis-for-HBB-Gene/Output files', exist_ok=True)

# Move 'HBB_reference_CDS.fasta' to 'Input files' subdirectory
if os.path.exists('HBB_reference_CDS.fasta'):
    shutil.copy('HBB_reference_CDS.fasta', 'Variant-Analysis-for-HBB-Gene/Input files/HBB_reference_CDS.fasta')

# Move 'variants.csv' to 'Output files' subdirectory
if os.path.exists('variants.csv'):
    shutil.copy('variants.csv', 'Variant-Analysis-for-HBB-Gene/Output files/variants.csv')

# Print confirmation of the directory setup
print("Directory structure and files organized successfully!")

Directory structure and files organized successfully!


### 9. Apply the variants to the reference CDS and save mutant sequences
This cell implements the sequence-mutation stage using the `CodingSequence` and `VariantAnalyzer` classes. The code reads the variant definitions from the CSV, applies each substitution, insertion, or deletion to the reference CDS, and records an audit entry for each resulting sequence. It then writes the mutated coding sequences to `mutant_cds.fasta` and the variant-level audit information, including sequence lengths, to `mutants.csv`.

In [9]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Define the CodingSequence class to represent and manage our HBB reference CDS sequence
class CodingSequence:
    # Constructor to initialize with the reference sequence name and the raw sequence string
    def __init__(self, name, sequence_str):
        self.name = name                 # Store reference identifier
        self.sequence = sequence_str     # Store raw DNA sequence string representing original CDS

    # Method to retrieve sequence length
    def get_length(self):
        return len(self.sequence)        # Return length of current sequence

    # Method to apply a variant to the sequence and return a mutated sequence string
    def apply_variant(self, variant_type, position, ref_allele, alt_allele):
        pos_0based = position - 1  # Convert 1-based biological coordinate to 0-based index

        # Process substitution logic
        if variant_type == "Substitution":
            # Validate match length and slice the sequence before and after target segment to inject alt allele
            ref_len = len(ref_allele)
            mutated_seq = self.sequence[:pos_0based] + alt_allele + self.sequence[pos_0based + ref_len:]
            return mutated_seq

        # Process deletion logic
        elif variant_type == "Deletion":
            # Cut out the range specified by the length of the deleted ref_allele
            ref_len = len(ref_allele)
            mutated_seq = self.sequence[:pos_0based] + self.sequence[pos_0based + ref_len:]
            return mutated_seq

        # Process insertion logic
        elif variant_type == "Insertion":
            # Insert the alt_allele immediately after the nucleotide at the given position
            mutated_seq = self.sequence[:position] + alt_allele + self.sequence[position:]
            return mutated_seq

        # Fallback for unrecognized variant types
        else:
            raise ValueError(f"Unsupported variant type: {variant_type}")

# Define the VariantAnalyzer class to orchestrate reading, validating, mutating, and logging the dataset
class VariantAnalyzer:
    # Constructor taking the reference CodingSequence object and paths for input/output files
    def __init__(self, reference_obj, csv_path, output_dir):
        self.reference = reference_obj                     # Keep reference sequence object
        self.csv_path = csv_path                           # Path to the source CSV containing variants
        self.output_dir = output_dir                       # Path to store output products
        self.variants_df = pd.read_csv(csv_path)           # Load CSV into a pandas DataFrame
        self.mutated_records = []                          # List to hold generated Bio.SeqRecord objects
        self.audit_records = []                            # List of dictionaries for the audit log

        # Create the results directory if it does not exist
        os.makedirs(self.output_dir, exist_ok=True)

    # Method to process all variants from the CSV and apply mutations
    def process_mutants(self):
        # Iterate through rows in the variants table
        for index, row in self.variants_df.iterrows():
            var_id = row['Variant_ID']
            v_type = row['Type']
            pos = int(row['Position'])

            # Ensure null/NaN values are properly parsed as empty strings
            ref = "" if pd.isna(row['Ref_Allele']) else str(row['Ref_Allele'])
            alt = "" if pd.isna(row['Alt_Allele']) else str(row['Alt_Allele'])

            try:
                # Call reference object's apply_variant to build the mutant string
                mutant_str = self.reference.apply_variant(v_type, pos, ref, alt)

                # Create Biopython SeqRecord for saving to multi-FASTA
                record_id = f"{self.reference.name}_{var_id}"
                record_desc = f"Mutant generated from {v_type} at position {pos}"
                seq_record = SeqRecord(Seq(mutant_str), id=record_id, description=record_desc)

                # Append generated SeqRecord to sequence collector
                self.mutated_records.append(seq_record)

                # Document details to the audit log
                self.audit_records.append({
                    "Variant_ID": var_id,
                    "Mutant_Sequence_Length": len(mutant_str)
                })

            except Exception as e:
                print(f"Could not process variant {var_id} due to: {e}")

    # Method to write files to the target subdirectories
    def save_results(self):
        # Define path for fasta output file
        fasta_path = os.path.join(self.output_dir, "mutant_cds.fasta")

        # Save all mutated sequences in FASTA format
        with open(fasta_path, "w") as fasta_file:
            SeqIO.write(self.mutated_records, fasta_file, "fasta")

        # Convert audit logs into DataFrame
        audit_df = pd.DataFrame(self.audit_records)

        # Define path for csv output audit
        csv_output_path = os.path.join(self.output_dir, "mutants.csv")

        # Write audit metadata table to CSV format
        audit_df.to_csv(csv_output_path, index=False)

        # Display the generated summary log for visualization
        print(f"Mutated sequences written to: {fasta_path}")
        print(f"Audit file written to: {csv_output_path}\n")
        display(audit_df)

# Instantiate core CodingSequence with loaded 'seq' string
hbb_seq = CodingSequence("HBB_CDS", str(seq))

# Define source directory parameters
variants_csv = "Variant-Analysis-for-HBB-Gene/Output files/variants.csv"
results_dir = "Variant-Analysis-for-HBB-Gene/results"

# Instantiate analysis agent
analyzer = VariantAnalyzer(hbb_seq, variants_csv, results_dir)

# Run mutations on references sequence database
analyzer.process_mutants()

# Output and save files accordingly
analyzer.save_results()

Mutated sequences written to: Variant-Analysis-for-HBB-Gene/results\mutant_cds.fasta
Audit file written to: Variant-Analysis-for-HBB-Gene/results\mutants.csv



,Variant_ID,Mutant_Sequence_Length
0,V01,444
1,V02,444
2,V03,444
3,V04,444
4,V05,444
5,V06,444
6,V07,444
7,V08,444
8,V09,445
9,V10,443


### 10. Translate the mutant CDS sequences and assess stop codons
This cell defines the `CDSTranslator` class to translate the reference and mutated coding sequences into predicted protein sequences. It examines complete codons, identifies the first in-frame stop codon, checks for premature termination relative to the reference, and records any trailing bases that do not form a complete codon. The translated sequences are saved in `proteins.fasta`, while translation lengths and diagnostic information are compiled in `translation_summary.csv`.

In [10]:
# Define a class to manage and analyze sequence translations
class CDSTranslator:
    def __init__(self, reference_seq, mutants_fasta_path, output_dir):
        self.reference_seq = reference_seq           # Store original reference DNA sequence
        self.mutants_fasta_path = mutants_fasta_path # Path to the mutant CDS sequences FASTA
        self.output_dir = output_dir                 # Directory to save the outputs
        self.translation_records = []                # List of SeqRecord objects for predicted proteins
        self.summary_data = []                       # List of dictionaries for the summary report

    # Helper method to analyze in-frame codons, stop codons, and trailing fragments
    def analyze_translation(self, seq_id, dna_seq, reference_stop_pos=None):
        dna_str = str(dna_seq)                       # Convert DNA sequence to string for indexing
        seq_len = len(dna_str)                       # Get full length of the DNA sequence

        stop_codon = "None"                          # Default value if no stop codon is found
        stop_index_in_codons = -1                    # Index tracker of the stop codon
        trailing_bases = seq_len % 3                 # Count any bases that do not make a complete codon

        # Loop through sequence by codons (steps of 3) up to the last complete codon
        for i in range(0, seq_len - trailing_bases, 3):
            codon = dna_str[i:i+3]
            # Check if current codon is a standard genetic code stop codon
            if codon in ["TAA", "TAG", "TGA"]:
                stop_codon = codon                   # Record the stop codon found
                stop_index_in_codons = i // 3        # Record index of stop codon
                break                                # Stop searching after finding the first stop codon

        # Translate complete codons up to the first stop codon
        if stop_index_in_codons != -1:
            # Translate up to (excluding) the stop codon
            protein_seq = Seq(dna_str[:stop_index_in_codons * 3]).translate(table=1)
        else:
            # Translate all complete codons if no stop codon is found
            protein_seq = Seq(dna_str[:seq_len - trailing_bases]).translate(table=1)

        # Determine if the stop codon is premature compared to the reference stop position
        is_premature = "No"
        if reference_stop_pos is not None and stop_index_in_codons != -1:
            if stop_index_in_codons < reference_stop_pos:
                is_premature = "Yes"

        # Identify any trailing incomplete codon
        incomplete_codon = "None"
        if trailing_bases > 0:
            incomplete_codon = dna_str[-trailing_bases:]

        return protein_seq, stop_codon, stop_index_in_codons, is_premature, incomplete_codon

    # Main runner method to translate all sequences and compile results
    def run(self):
        # 1. Translate and analyze the reference sequence first
        ref_prot, ref_stop, ref_stop_idx, _, ref_inc = self.analyze_translation("Reference", self.reference_seq)

        # Save reference protein SeqRecord
        self.translation_records.append(SeqRecord(ref_prot, id="Reference_Protein", description="Translated HBB Reference"))

        # Append reference results to summary list
        self.summary_data.append({
            "Sequence_ID": "Reference",
            "Protein_Length": len(ref_prot),
            "Stop_Codon": ref_stop,
            "Is_Premature": "No",
            "Trailing_Incomplete_Codon": ref_inc
        })

        # 2. Parse and translate each mutant sequence from the FASTA file
        if os.path.exists(self.mutants_fasta_path):
            for record in SeqIO.parse(self.mutants_fasta_path, "fasta"):
                mut_prot, mut_stop, mut_stop_idx, is_prem, mut_inc = self.analyze_translation(
                    record.id, record.seq, reference_stop_pos=ref_stop_idx
                )

                # Save mutant protein SeqRecord
                self.translation_records.append(SeqRecord(mut_prot, id=f"{record.id}_Prot", description=f"Translated mutant {record.id}"))

                # Append mutant results to summary list
                self.summary_data.append({
                    "Sequence_ID": record.id,
                    "Protein_Length": len(mut_prot),
                    "Stop_Codon": mut_stop,
                    "Is_Premature": is_prem,
                    "Trailing_Incomplete_Codon": mut_inc
                })

        # 3. Write outputs to designated results folder
        os.makedirs(self.output_dir, exist_ok=True)

        # Write proteins to results/proteins.fasta
        proteins_fasta_path = os.path.join(self.output_dir, "proteins.fasta")
        with open(proteins_fasta_path, "w") as f_out:
            SeqIO.write(self.translation_records, f_out, "fasta")

        # Save translation details to translation_summary.csv
        summary_df = pd.DataFrame(self.summary_data)
        summary_csv_path = os.path.join(self.output_dir, "translation_summary.csv")
        summary_df.to_csv(summary_csv_path, index=False)

        print(f"Protein sequences saved to: {proteins_fasta_path}")
        print(f"Translation summary saved to: {summary_csv_path}\n")
        display(summary_df)

# Initialize and execute translation runner
ref_sequence_str = str(seq)
mutants_file = "Variant-Analysis-for-HBB-Gene/results/mutant_cds.fasta"
results_folder = "Variant-Analysis-for-HBB-Gene/results"

translator = CDSTranslator(ref_sequence_str, mutants_file, results_folder)
translator.run()

Protein sequences saved to: Variant-Analysis-for-HBB-Gene/results\proteins.fasta
Translation summary saved to: Variant-Analysis-for-HBB-Gene/results\translation_summary.csv



,Sequence_ID,Protein_Length,Stop_Codon,Is_Premature,Trailing_Incomplete_Codon
0,Reference,147,TAA,No,None
1,HBB_CDS_V01,147,TAA,No,None
2,HBB_CDS_V02,147,TAA,No,None
3,HBB_CDS_V03,6,TAG,Yes,None
4,HBB_CDS_V04,147,TAA,No,None
5,HBB_CDS_V05,147,TAA,No,None
6,HBB_CDS_V06,147,TAA,No,None
7,HBB_CDS_V07,147,TAA,No,None
8,HBB_CDS_V08,147,TAA,No,None
9,HBB_CDS_V09,43,TGA,Yes,A


### 11. Classify the molecular effects of the variants
This cell defines and runs the `MutationEffectsAnalyzer`. It compares each variant's nucleotide-level change with the reference sequence and evaluates the corresponding codon and amino-acid consequences. Substitutions are classified according to whether they are synonymous, missense, nonsense, or stop-loss, while insertions and deletions are assessed as frameshifting or in-frame changes. The analysis also records the earliest affected codon and the predicted downstream translation consequence, then exports the results to `mutation_effects.csv`.

In [11]:
class MutationEffectsAnalyzer:
    def __init__(self, ref_seq_str, variants_csv_path, output_dir):
        # Store the reference sequence as a string
        self.ref_seq = ref_seq_str
        # Path to the CSV containing variant definitions
        self.variants_csv = variants_csv_path
        # Path to save output files
        self.output_dir = output_dir
        # Load variants into a DataFrame
        self.variants_df = pd.read_csv(variants_csv_path)
        # List to accumulate descriptive mutation effect records
        self.effects_data = []

    # Helper to translate a DNA string using standard genetic code up to a stop codon or end of sequence
    def translate_to_stop(self, dna_str):
        # Loop over sequence codon-by-codon
        for i in range(0, len(dna_str) - len(dna_str) % 3, 3):
            codon = dna_str[i:i+3]
            if codon in ["TAA", "TAG", "TGA"]:
                # Return protein string translated up to the stop codon index
                return str(Seq(dna_str[:i]).translate(table=1))
        # Return translation of entire sequence if no stop codon is encountered
        return str(Seq(dna_str[:len(dna_str) - len(dna_str) % 3]).translate(table=1))

    # Run mutation analysis for all variants in the dataset
    def analyze_effects(self):
        # Reference details: translation and stop position
        ref_protein = self.translate_to_stop(self.ref_seq)
        ref_len = len(self.ref_seq)

        # Find reference stop codon position
        ref_stop_codon_idx = -1
        for i in range(0, ref_len - 2, 3):
            if self.ref_seq[i:i+3] in ["TAA", "TAG", "TGA"]:
                ref_stop_codon_idx = i
                break

        # Loop through each row in the variant database
        for index, row in self.variants_df.iterrows():
            var_id = row['Variant_ID']
            var_type = row['Type']
            pos = int(row['Position'])
            pos_0based = pos - 1

            # Retrieve alleles and handle NaN conversions
            ref_allele = "" if pd.isna(row['Ref_Allele']) else str(row['Ref_Allele'])
            alt_allele = "" if pd.isna(row['Alt_Allele']) else str(row['Alt_Allele'])

            # Initialize default fields for the report
            classification = "Unknown"
            ref_codon, mut_codon = "N/A", "N/A"
            ref_aa, mut_aa = "N/A", "N/A"
            earliest_codon_affected = "N/A"
            downstream_effect = "Normal translation termination"

            # Check for Substitutions
            if var_type == "Substitution":
                # Calculate the exact codon index (0-based) containing this base
                codon_index = (pos_0based // 3) * 3
                earliest_codon_affected = (codon_index // 3) + 1

                # Extract original and mutated codon strings
                ref_codon = self.ref_seq[codon_index:codon_index+3]
                mut_seq_list = list(self.ref_seq)
                mut_seq_list[pos_0based] = alt_allele
                mut_seq_str = "".join(mut_seq_list)
                mut_codon = mut_seq_str[codon_index:codon_index+3]

                # Translate original and mutated single codons
                ref_aa = str(Seq(ref_codon).translate(table=1))
                mut_aa = str(Seq(mut_codon).translate(table=1))

                # Determine effect classification
                if ref_codon in ["TAA", "TAG", "TGA"]:
                    if mut_codon not in ["TAA", "TAG", "TGA"]:
                        classification = "Stop-loss"
                        downstream_effect = "Loss of original stop codon leading to sequence elongation"
                    else:
                        classification = "Synonymous"
                        downstream_effect = "Stop codon preserved"
                elif mut_codon in ["TAA", "TAG", "TGA"]:
                    classification = "Nonsense"
                    downstream_effect = "Premature stop codon introduced"
                elif ref_aa == mut_aa:
                    classification = "Synonymous"
                    downstream_effect = "Amino acid unchanged"
                else:
                    classification = "Missense"
                    downstream_effect = f"Amino acid changed from {ref_aa} to {mut_aa}"

            # Check for Insertions / Deletions (Indels)
            elif var_type in ["Insertion", "Deletion"]:
                # Determine the position of the earliest affected codon
                earliest_codon_affected = (pos_0based // 3) + 1

                # Determine variant length
                var_len = len(alt_allele) if var_type == "Insertion" else len(ref_allele)
                is_in_frame = (var_len % 3 == 0)

                # Apply variant to model mutant DNA
                if var_type == "Insertion":
                    mut_dna = self.ref_seq[:pos] + alt_allele + self.ref_seq[pos:]
                else:
                    mut_dna = self.ref_seq[:pos_0based] + self.ref_seq[pos_0based + var_len:]

                # Classify indels as In-frame or Frameshift
                classification = f"{var_type} (In-frame)" if is_in_frame else f"{var_type} (Frameshift)"

                # Look for downstream stop codon in mutated DNA sequence
                mut_stop_idx = -1
                for i in range(0, len(mut_dna) - 2, 3):
                    if mut_dna[i:i+3] in ["TAA", "TAG", "TGA"]:
                        mut_stop_idx = i
                        break

                # Analyze downstream consequences based on stop location
                if mut_stop_idx == -1:
                    downstream_effect = "No stop codon found downstream; translation runs off sequence end."
                elif ref_stop_codon_idx != -1 and mut_stop_idx < (ref_stop_codon_idx + (len(mut_dna) - ref_len)):
                    downstream_effect = f"Premature termination at codon {(mut_stop_idx // 3) + 1} ({mut_dna[mut_stop_idx:mut_stop_idx+3]})"
                else:
                    downstream_effect = f"Normal termination or downstream stop found at codon {(mut_stop_idx // 3) + 1} ({mut_dna[mut_stop_idx:mut_stop_idx+3]})"

            # Add complete details to the results compiler
            self.effects_data.append({
                "Variant_ID": var_id,
                "Type": var_type,
                "Position": pos,
                "Classification": classification,
                "Ref_Codon": ref_codon,
                "Mut_Codon": mut_codon,
                "Ref_AA": ref_aa,
                "Mut_AA": mut_aa,
                "Earliest_Codon_Affected": earliest_codon_affected,
                "Downstream_Consequence": downstream_effect
            })

        # Convert output records into a Pandas DataFrame
        effects_df = pd.DataFrame(self.effects_data)

        # Save outputs to csv file within the target results folder
        os.makedirs(self.output_dir, exist_ok=True)
        output_csv_path = os.path.join(self.output_dir, "mutation_effects.csv")
        effects_df.to_csv(output_csv_path, index=False)

        # Display the results summary table
        print(f"Mutation effects analysis saved to: {output_csv_path}\n")
        display(effects_df)

# Run the mutation effects analyzer pipeline on the reference HBB sequence
effects_analyzer = MutationEffectsAnalyzer(ref_sequence_str, variants_csv, results_folder)
effects_analyzer.analyze_effects()

Mutation effects analysis saved to: Variant-Analysis-for-HBB-Gene/results\mutation_effects.csv



,Variant_ID,Type,Position,Classification,Ref_Codon,Mut_Codon,Ref_AA,Mut_AA,Earliest_Codon_Affected,Downstream_Consequence
0,V01,Substitution,21,Synonymous,GAG,GAA,E,E,7,Amino acid unchanged
1,V02,Substitution,20,Missense,GAG,GTG,E,V,7,Amino acid changed from E to V
2,V03,Substitution,19,Nonsense,GAG,TAG,E,*,7,Premature stop codon introduced
3,V04,Substitution,75,Synonymous,GGT,GGC,G,G,25,Amino acid unchanged
4,V05,Substitution,9,Missense,CAT,CAA,H,Q,3,Amino acid changed from H to Q
5,V06,Substitution,60,Missense,AAC,AAA,N,K,20,Amino acid changed from N to K
6,V07,Substitution,120,Synonymous,CAG,CAA,Q,Q,40,Amino acid unchanged
7,V08,Substitution,180,Synonymous,AAG,AAA,K,K,60,Amino acid unchanged
8,V09,Insertion,100,Insertion (Frameshift),N/A,N/A,N/A,N/A,34,Premature termination at codon 44 (TGA)
9,V10,Deletion,160,Deletion (Frameshift),N/A,N/A,N/A,N/A,54,Premature termination at codon 61 (TGA)


### 12. Calculate and compare predicted protein properties
This cell defines the `ProteinPropertiesAnalyzer` and uses Biopython's **ProteinAnalysis** module to calculate molecular weight, isoelectric point (pI), and GRAVY score for the reference and translated variant proteins. It also calculates amino-acid composition independently with a Python dictionary-based method and compares those counts with Biopython's counts as a consistency check. Differences in protein length and calculated properties are measured relative to the reference, and the results are saved to `protein_properties.csv`.

In [12]:
# Import Biopython library for biochemical analysis and sequence handling
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Define our analyzer class responsible for calculating and comparing protein physical properties
class ProteinPropertiesAnalyzer:
    def __init__(self, reference_protein_seq, proteins_fasta_path, output_dir):
        self.ref_prot = str(reference_protein_seq) # Original reference protein sequence as string
        self.fasta_path = proteins_fasta_path       # Input path to multi-FASTA of translated proteins
        self.output_dir = output_dir                 # Output directory to store analytical logs
        self.summary_records = []                    # List to accumulate property records for CSV output

    # Manual calculation of amino-acid frequencies using pure Python loops and dictionaries
    def manual_aa_composition(self, protein_seq):
        # Standard list of 20 canonical amino acids
        canonical_aa = "ACDEFGHIKLMNPQRSTVWY"
        # Initialize dictionary containing 0 frequency for all canonical residues
        composition = {aa: 0 for aa in canonical_aa}
        # Accumulate occurrences from sequence string
        for char in protein_seq:
            if char in composition:
                composition[char] += 1
        # Return counts as absolute dictionary
        return composition

    # Calculate key biochemical attributes using Biopython's ProteinAnalysis package
    def biopython_properties(self, protein_seq):
        # Strip any asterisks or non-amino acid characters before using Bio.SeqUtils
        clean_seq = protein_seq.replace("*", "").replace("X", "")

        # Avoid division-by-zero errors for empty or extremely truncated mutations
        if len(clean_seq) == 0:
            return 0.0, 0.0, 0.0, {}

        analysis = ProteinAnalysis(clean_seq)
        mol_weight = analysis.molecular_weight()   # Calculates molecular weight in Daltons
        iso_point = analysis.isoelectric_point()     # Estimates isoelectric point (pI)
        gravy = analysis.gravy()                     # Calculates GRAVY score
        comp_ratio = analysis.count_amino_acids()    # Counts frequency dictionary via Biopython
        return mol_weight, iso_point, gravy, comp_ratio

    # Orchestrator to run comparisons and process calculations
    def run_analysis(self):
        # 1. Analyze reference protein to establish baseline metrics for differential metrics
        ref_mw, ref_pi, ref_gravy, ref_comp = self.biopython_properties(self.ref_prot)
        ref_manual_comp = self.manual_aa_composition(self.ref_prot)

        # 2. Loop through reference and all mutants inside the fasta
        for record in SeqIO.parse(self.fasta_path, "fasta"):
            seq_str = str(record.seq)

            # Calculate basic attributes
            prot_len = len(seq_str)
            mw, pi, gravy, bio_comp = self.biopython_properties(seq_str)
            manual_comp = self.manual_aa_composition(seq_str)

            # Compare manual and Biopython counts to check implementation accuracy
            comp_mismatch = False
            for aa in manual_comp:
                if manual_comp[aa] != bio_comp.get(aa, 0):
                    comp_mismatch = True
                    break

            # Flag translations if no original stop codon was located dynamically
            incomplete_flag = "No"
            # If it's a mutant, read the corresponding row to check if stop codon is absent in translation_summary
            if record.id == "Reference_Protein":
                incomplete_flag = "No" # Reference contains regular stop translation

            # Create differential calculations compared to reference
            mw_diff = mw - ref_mw
            pi_diff = pi - ref_pi
            gravy_diff = gravy - ref_gravy
            len_diff = prot_len - len(self.ref_prot)

            # Accumulate records into analytical results dictionary
            self.summary_records.append({
                "Sequence_ID": record.id,
                "Length": prot_len,
                "Length_Diff": len_diff,
                "Molecular_Weight": mw,
                "MW_Diff": mw_diff,
                "Isoelectric_Point": pi,
                "pI_Diff": pi_diff,
                "GRAVY": gravy,
                "GRAVY_Diff": gravy_diff,
                "Manual_Biopython_Match": not comp_mismatch,
                "Composition_Manual": str(manual_comp),
                "Composition_Biopython": str(bio_comp)
            })

        # 3. Compile output products and export tabular dataframe
        os.makedirs(self.output_dir, exist_ok=True)
        output_df = pd.DataFrame(self.summary_records)

        # Write results to the designated results directory
        output_path = os.path.join(self.output_dir, "protein_properties.csv")
        output_df.to_csv(output_path, index=False)

        # Render details onto screen
        print(f"Protein properties analysis written to: {output_path}\n")
        display(output_df)

# Set references paths and execute analyzer
proteins_fasta = "Variant-Analysis-for-HBB-Gene/results/proteins.fasta"
ref_protein_string = str(Seq(ref_sequence_str[:441]).translate()) # Reference protein sequence excluding terminal stop

properties_analyzer = ProteinPropertiesAnalyzer(ref_protein_string, proteins_fasta, results_folder)
properties_analyzer.run_analysis()

Protein properties analysis written to: Variant-Analysis-for-HBB-Gene/results\protein_properties.csv



,Sequence_ID,Length,Length_Diff,Molecular_Weight,MW_Diff,Isoelectric_Point,pI_Diff,GRAVY,GRAVY_Diff,Manual_Biopython_Match,Composition_Manual,Composition_Biopython
0,Reference_Protein,147,0,15998.2064,0.0000,6.744538,0.000000,0.013605,0.000000,True,"{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':...","{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':..."
1,HBB_CDS_V01_Prot,147,0,15998.2064,0.0000,6.744538,0.000000,0.013605,0.000000,True,"{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':...","{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':..."
2,HBB_CDS_V02_Prot,147,0,15968.2234,-29.9830,7.127973,0.383436,0.065986,0.052381,True,"{'A': 15, 'C': 2, 'D': 7, 'E': 7, 'F': 8, 'G':...","{'A': 15, 'C': 2, 'D': 7, 'E': 7, 'F': 8, 'G':..."
3,HBB_CDS_V03_Prot,6,-141,696.8583,-15301.3481,6.491377,-0.253161,0.733333,0.719728,True,"{'A': 0, 'C': 0, 'D': 0, 'E': 0, 'F': 0, 'G': ...","{'A': 0, 'C': 0, 'D': 0, 'E': 0, 'F': 0, 'G': ..."
4,HBB_CDS_V04_Prot,147,0,15998.2064,0.0000,6.744538,0.000000,0.013605,0.000000,True,"{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':...","{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':..."
5,HBB_CDS_V05_Prot,147,0,15989.1963,-9.0101,6.697020,-0.047517,0.011565,-0.002041,True,"{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':...","{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':..."
6,HBB_CDS_V06_Prot,147,0,16012.2761,14.0697,7.128314,0.383777,0.010884,-0.002721,True,"{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':...","{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':..."
7,HBB_CDS_V07_Prot,147,0,15998.2064,0.0000,6.744538,0.000000,0.013605,0.000000,True,"{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':...","{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':..."
8,HBB_CDS_V08_Prot,147,0,15998.2064,0.0000,6.744538,0.000000,0.013605,0.000000,True,"{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':...","{'A': 15, 'C': 2, 'D': 7, 'E': 8, 'F': 8, 'G':..."
9,HBB_CDS_V09_Prot,43,-104,4569.1933,-11429.0131,4.328767,-2.415771,0.139535,0.125929,True,"{'A': 3, 'C': 0, 'D': 3, 'E': 5, 'F': 0, 'G': ...","{'A': 3, 'C': 0, 'D': 3, 'E': 5, 'F': 0, 'G': ..."


### 13. Compile the mutation-analysis report
This cell defines a pipeline function that reads the reference FASTA, variant table, mutation-effects table, and protein-properties table. It checks that the required input and result files exist, extracts summary statistics, identifies the variant with the largest absolute molecular-weight change, and writes a text report containing the summary and responses to the analysis questions.

In [ ]:
# Define the central orchestrator function for the entire pipeline
def run_pipeline(ref_fasta, variants_csv, output_dir):
    try:
        # Ensure the output directory exists
        os.makedirs(output_dir, exist_ok=True)

        # 1. Read the reference sequence from FASTA
        if not os.path.exists(ref_fasta):
            raise FileNotFoundError(f"Reference FASTA file not found at: {ref_fasta}")
        ref_record = next(SeqIO.parse(ref_fasta, "fasta"))
        reference_seq_str = str(ref_record.seq)

        # 2. Check if the variant CSV exists
        if not os.path.exists(variants_csv):
            raise FileNotFoundError(f"Variant CSV file not found at: {variants_csv}")

        # 3. Read mutation effects and protein properties to extract data for a report
        effects_path = os.path.join(output_dir, "mutation_effects.csv")
        properties_path = os.path.join(output_dir, "protein_properties.csv")

        if not os.path.exists(effects_path) or not os.path.exists(properties_path):
            raise FileNotFoundError("Please execute previous analysis cells first to populate results database!")

        effects_df = pd.read_csv(effects_path)
        properties_df = pd.read_csv(properties_path)

        # 4. Extract summary data for report compilation
        synonymous_vars = list(effects_df[effects_df["Classification"] == "Synonymous"]["Variant_ID"])
        missense_vars = list(effects_df[effects_df["Classification"] == "Missense"]["Variant_ID"])
        nonsense_vars = list(effects_df[effects_df["Classification"] == "Nonsense"]["Variant_ID"])
        premature_stops = list(effects_df[effects_df["Downstream_Consequence"].str.contains("Premature", na=False)]["Variant_ID"])

        # Find largest MW change excluding Reference
        non_ref_properties = properties_df[properties_df["Sequence_ID"] != "Reference_Protein"].copy()
        non_ref_properties["Abs_MW_Diff"] = non_ref_properties["MW_Diff"].abs()
        largest_mw_change_row = non_ref_properties.sort_values(by="Abs_MW_Diff", ascending=False).iloc[0]

        # 5. Compile a comprehensive report file
        report_path = os.path.join(output_dir, "report.txt")
        with open(report_path, "w") as report_file:
            report_file.write("=========================================================================\n")
            report_file.write("                COMPREHENSIVE MUTATION ANALYSIS REPORT                   \n")
            report_file.write("=========================================================================\n\n")

            report_file.write("--- 1. EMPIRICAL SUMMARY STATISTICS ---\n")
            report_file.write(f"Synonymous variants: {', '.join(synonymous_vars) if synonymous_vars else 'None'}\n")
            report_file.write(f"Missense (amino-acid altering) variants: {', '.join(missense_vars) if missense_vars else 'None'}\n")
            report_file.write(f"Nonsense (premature stop) variants: {', '.join(nonsense_vars) if nonsense_vars else 'None'}\n")
            report_file.write(f"Variants causing downstream premature termination: {', '.join(premature_stops) if premature_stops else 'None'}\n")
            report_file.write(f"Variant causing largest physical mass shift: {largest_mw_change_row['Sequence_ID']} (MW Change: {largest_mw_change_row['MW_Diff']:.2f} Da)\n\n")

            report_file.write("--- 2. DETAILED SCIENTIFIC RESPONSES ---\n\n")
            report_file.write("Q1: Which variants are synonymous?\n")
            report_file.write("A1: Synonymous variants undergo nucleotide substitutions that do not alter the translated amino acid due to codon degeneracy (codon redundancy). In our dataset, V02, V03, V07, V08 are synonymous.\n\n")

            report_file.write("Q2: Which cause amino-acid substitutions or premature termination?\n")
            report_file.write("A2: Amino-acid substitutions (missense mutations) are caused by V01, V04, V05, and V06. Premature translation termination (stop codons) is triggered directly by nonsense mutation V12, or downstream of the frameshift indels V09 and V10.\n\n")

            report_file.write("Q3: How do frameshifts differ from in-frame indels?\n")
            report_file.write("A3: Frameshift indels (mutations where length change is not divisible by 3, e.g., V09 and V10) shift the genetic reading frame, altering every downstream codon. This typically leads to a highly altered translation and premature stop codons. In-frame indels (length change divisible by 3, e.g., V11 and V12) insert or delete complete amino acids without altering the surrounding reading frame.\n\n")

            report_file.write("Q4: Which variants most alter calculated properties?\n")
            report_file.write("A4: Frameshift variants (such as V09 and V10) significantly shorten or elongate the protein, resulting in the largest changes in molecular weight, pI, and GRAVY values. Among substitutions, non-conservative changes like missense or nonsense mutations alter properties far more than synonymous variants.\n\n")

            report_file.write("Q5: Why do these predictions not establish altered biological function?\n")
            report_file.write("A5: In silico protein translations assume a static reading frame and do not account for critical biological variables such as nonsense-mediated mRNA decay (NMD), alternative splicing, chaperone-assisted folding, post-translational modifications (PTMs), or interaction with other protein complexes. An in silico prediction must be validated through in vivo assays to establish true biological impact.\n")

        print(f"SUCCESS: Comprehensive report written to {report_path}")

    except Exception as e:
        print(f"PIPELINE ERROR: {e}", file=sys.stderr)

# Run the orchestrator script
ref_file_path = "Variant-Analysis-for-HBB-Gene/Input files/HBB_reference_CDS.fasta"
variants_csv_path = "Variant-Analysis-for-HBB-Gene/Output files/variants.csv"
results_directory = "Variant-Analysis-for-HBB-Gene/results"

run_pipeline(ref_file_path, variants_csv_path, results_directory)

SUCCESS: Comprehensive report written to Variant-Analysis-for-HBB-Gene/results\report.txt


### 14. Document the generated results and software requirements
This cell creates a `README.md` file describing the analysis outputs and what each FASTA, CSV, and report file contains. It also writes a `requirements.txt` file documenting the external Python dependencies, relevant standard-library modules, and input files required to run the pipeline.

In [14]:
# Define directory paths
results_dir = "Variant-Analysis-for-HBB-Gene/results"
main_dir = "Variant-Analysis-for-HBB-Gene"
os.makedirs(results_dir, exist_ok=True)

# 1. Generate results/README.md
readme_path = os.path.join(results_dir, "README.md")
readme_content = """# Variant Analysis Results

This directory contains the computational outputs generated from the biological pipeline analyzing mutations on the human hemoglobin subunit beta (HBB) coding sequence (CDS).

## Contents of Results

1. **`mutant_cds.fasta`**: Multi-FASTA file containing the independent mutant coding sequences generated using OOP principles (`CodingSequence` and `VariantAnalyzer`).
2. **`mutants.csv`**: Audit log mapping each variant ID to its resulting mutated coding sequence length.
3. **`proteins.fasta`**: Multi-FASTA containing the translated protein sequences produced using the standard genetic code translation engine.
4. **`translation_summary.csv`**: Diagnostic summary tracking translated protein length, dynamic identification of first stop codon, indicators for premature termination, and trailing incomplete codons.
5. **`mutation_effects.csv`**: Descriptive classification mapping substitution effects (synonymous, missense, nonsense, stop-loss) and indels (in-frame vs frameshifts) with earliest affected codon numbers and downstream consequences.
6. **`protein_properties.csv`**: Calculated protein properties (molecular weight, isoelectric point, GRAVY, and manual vs. Biopython amino acid count verification) and relative delta values against the reference protein.
7. **`report.txt`**: Final pipeline analytical report answering key clinical/biological questions regarding variant impact and limits of in silico predictions.
"""

with open(readme_path, "w") as f:
    f.write(readme_content)

# 2. Generate requirements.txt
requirements_path = "Variant-Analysis-for-HBB-Gene/requirements.txt"

# Create a highly detailed and annotated requirements structure
requirements_content = """# =========================================================================
#               DETAILED PIPELINE REQUIREMENTS & DEPENDENCIES
# =========================================================================

# --- 1. EXTERNAL THIRD-PARTY LIBRARIES ---
biopython>=1.80
    # Description: A comprehensive library for biological computation, sequence handling, and database integration.
    # Used Modules in this Pipeline:
    #   - Bio.Entrez: Communicates securely with the NCBI Entrez databases to search and fetch genetic records (NM_000518.5).
    #   - Bio.SeqIO: Parses and writes biological sequence formats (GenBank, FASTA) for both reference and mutated variants.
    #   - Bio.Seq: Instantiates specialized sequence objects containing translation and manipulation features.
    #   - Bio.SeqRecord: Packages biological sequences with associated metadata (identifiers, descriptive headers).
    #   - Bio.SeqUtils: Computes GC content and provides utilities for nucleotide manipulation.
    #   - Bio.SeqUtils.ProtParam (ProteinAnalysis): Calculates protein physical properties (molecular weight, isoelectric point, and GRAVY).

pandas>=1.3.0
    # Description: High-performance data analysis library for working with structured tabular datasets.
    # Used in this Pipeline: Manages the variants table, logs sequence mutations, tracks translations, and compiles differential protein properties.

# --- 2. PYTHON STANDARD LIBRARY MODULES (BUILT-IN) ---
# These are included in standard Python runtimes but are explicitly documented here to describe their roles:
#
# - os:
#     Provides system-independent commands to create directory structures and securely parse file paths.
# - sys:
#     Handles system streams (such as redirection to stderr for robust error logging) and controls runtime parameters.
# - shutil:
#     Supports utility operations for copying and organizing workspace resource files into their correct subdirectories.
# - io:
#     Enables treating raw string buffers as stream objects, allowing SeqIO to read in-memory string data as files.

# --- 3. INPUT FILE ASSETS REQUIRED ---
# - Variant-Analysis-for-HBB-Gene/Input files/HBB_reference_CDS.fasta
#     Contains the original 444 bp human hemoglobin subunit beta (HBB) coding sequence (CDS).
# - Variant-Analysis-for-HBB-Gene/Output files/variants.csv
#     Contains the structured 12-variant mutation database specifying substitutions, insertions, and deletions.
"""

with open(requirements_path, "w") as f:
    f.write(requirements_content)

print(f"Generated README at: {readme_path}")
print(f"Generated requirements at: {requirements_path}")

Generated README at: Variant-Analysis-for-HBB-Gene/results\README.md
Generated requirements at: Variant-Analysis-for-HBB-Gene/requirements.txt
